# NRC-VAD Valence Sentiment

This notebook estimates annual connotational sentiment for ADHD, Autism, and the three baseline terms. It uses the shared LSC mention-context table, extracts Baes-style local collocates from the ±5-token windows, matches them to NRC-VAD v2.1, and reports annual valence trajectories with document-level bootstrap confidence intervals.

## Setup

The notebook uses publication year (`lsc_year`) from the shared context table as the diachronic axis. It requires NRC-VAD v2.1 under `data/external` and the spaCy `en_core_web_sm` model for consistent tokenisation and lemmatisation. The VAD collocate table is saved under `data/interim/lsc/vad` so the later Intensity stage can reuse the same preprocessing.

In [10]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
NRC_VAD_PATH = PROJECT_ROOT / "data/external/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt"
INTERIM_VAD_DIR = PROJECT_ROOT / "data/interim/lsc/vad"
SENTIMENT_DIR = PROJECT_ROOT / "data/processed/lsc/sentiment"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/sentiment"

INTERIM_VAD_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VAD_MATCHES_PATH = INTERIM_VAD_DIR / "lsc_vad_collocate_matches.parquet"
VAD_CONTEXT_COVERAGE_PATH = INTERIM_VAD_DIR / "lsc_vad_context_coverage.parquet"
ANNUAL_VALENCE_PATH = SENTIMENT_DIR / "lsc_sentiment_annual_valence.csv"
COVERAGE_PATH = SENTIMENT_DIR / "lsc_sentiment_coverage.csv"
TOP_COLLOCATES_PATH = SENTIMENT_DIR / "lsc_sentiment_top_collocates.csv"
AUDIT_FLAGS_PATH = SENTIMENT_DIR / "lsc_sentiment_audit_flags.csv"
VALENCE_PLOT_PATH = FIGURE_DIR / "lsc_sentiment_valence_trajectories.png"
COVERAGE_PLOT_PATH = FIGURE_DIR / "lsc_sentiment_coverage.png"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_YEARS = list(range(2014, 2027))
BOOTSTRAP_REPETITIONS = 500
BOOTSTRAP_SEED = 123
LOW_MATCHED_TOKEN_COVERAGE_WARN = 0.35
LOW_CONTEXT_COVERAGE_WARN = 0.50
TOP_COLLOCATE_SHARE_WARN = 0.20

LSC_UNIT_LABELS = {
    "ADHD": "ADHD",
    "Autism": "Autism",
    "frustration": "Frustration",
    "loneliness": "Loneliness",
    "sadness": "Sadness",
}
LSC_UNIT_COLORS = {
    "ADHD": "#0072B2",
    "Autism": "#D55E00",
    "frustration": "#009E73",
    "loneliness": "#CC79A7",
    "sadness": "#6E6E6E",
}
LSC_UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
LSC_FIGURE_DPI = 300

assert CONTEXT_PATH.exists(), CONTEXT_PATH
assert NRC_VAD_PATH.exists(), NRC_VAD_PATH

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError as exc:
    raise OSError(
        "Missing spaCy model en_core_web_sm. Install it with: python -m spacy download en_core_web_sm"
    ) from exc

PROJECT_ROOT


PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak')

## Load Inputs

Only the shared context columns needed for sentiment are loaded. The annual axis is `lsc_year`; `source_year` is retained for provenance checks but is not used for the sentiment trajectory.

In [11]:
context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "collapsed_raw_forms",
    "collapsed_matched_texts",
    "registered_domain",
    "token_window_5",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts["context_row_id"] = contexts.index.astype(int)

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")
if contexts["token_window_5"].fillna("").str.strip().eq("").any():
    raise RuntimeError("Some context rows have empty token_window_5 values.")

vad = pd.read_csv(NRC_VAD_PATH, sep="	")
expected_vad_columns = {"term", "valence", "arousal", "dominance"}
if set(vad.columns) != expected_vad_columns:
    raise RuntimeError(f"Unexpected NRC-VAD columns: {vad.columns.tolist()}")
for column in ["valence", "arousal", "dominance"]:
    if not vad[column].between(-1, 1).all():
        raise RuntimeError(f"NRC-VAD {column} values outside expected [-1, 1] range.")

print(f"Contexts: {len(contexts):,}")
print(f"Documents: {contexts['doc_id'].nunique():,}")
print(f"NRC-VAD terms: {len(vad):,}")
contexts.head()

Contexts: 146,471
Documents: 106,045
NRC-VAD terms: 54,801


,doc_id,lsc_year,published_year,source_year,analysis_unit,term_role,target_group,raw_form,matched_text,collapsed_raw_forms,collapsed_matched_texts,registered_domain,token_window_5,context_row_id
0,00244c78408b04c9,2014,2014,2014,ADHD,target,ADHD,adhd,ADHD,adhd,ADHD,ifsw.org,increase of medications to treat ADHD in children. ”Medications,0
1,002fdb2e9d610482,2014,2014,2024,ADHD,target,ADHD,adhd,ADHD,adhd,ADHD,agrinews24.com,been used for children with ADHD. A key goal is,1
2,0063af4493275c3e,2014,2014,2019,ADHD,target,ADHD,adhd,ADHD,adhd,ADHD,thebullandbear.com,"support moisturizers for NAION, ADHD as the cycle of '",2
3,0063af4493275c3e,2014,2014,2019,ADHD,target,ADHD,adhd,ADHD,adhd,ADHD,thebullandbear.com,Buy with a fab condition ADHD again to slimming any right,3
4,0063af4493275c3e,2014,2014,2019,ADHD,target,ADHD,adhd,ADHD,adhd,ADHD,thebullandbear.com,. observed soap the Relative ADHD. I was my exercise,4


## Prepare NRC-VAD Lookup

NRC-VAD v2.1 includes unigrams and multi-word expressions. To use lemmatised context windows consistently, lexicon terms are also tokenised and lemmatised with the same spaCy model. If multiple surface entries collapse to the same lemma phrase, their VAD scores are averaged.

In [12]:
lexicon_records: list[dict[str, object]] = []
vad_terms = vad["term"].fillna("").map(str).tolist()
for doc, row in zip(nlp.pipe(vad_terms, batch_size=1000), vad.itertuples(index=False)):
    lemma_tokens = [token.lemma_.lower() for token in doc if token.is_alpha and len(token.lemma_) > 1]
    if not lemma_tokens:
        continue
    lexicon_records.append(
        {
            "lexicon_key": " ".join(lemma_tokens),
            "lexicon_tuple": tuple(lemma_tokens),
            "lexicon_length": len(lemma_tokens),
            "term": row.term,
            "valence": float(row.valence),
            "arousal": float(row.arousal),
            "dominance": float(row.dominance),
        }
    )

lexicon = pd.DataFrame(lexicon_records)
lexicon_lookup = (
    lexicon.groupby(["lexicon_key", "lexicon_length"], as_index=False)
    .agg(
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        dominance=("dominance", "mean"),
        source_terms=("term", lambda values: " | ".join(sorted(set(values))[:5])),
        source_term_count=("term", "nunique"),
    )
)
lexicon_lookup["lexicon_tuple"] = lexicon_lookup["lexicon_key"].str.split().map(tuple)

unigram_lookup = {
    row.lexicon_tuple: row
    for row in lexicon_lookup.loc[lexicon_lookup["lexicon_length"] == 1].itertuples(index=False)
}
mwe_lookup = {
    row.lexicon_tuple: row
    for row in lexicon_lookup.loc[lexicon_lookup["lexicon_length"] > 1].itertuples(index=False)
}
mwe_lengths = sorted({len(key) for key in mwe_lookup}, reverse=True)

print(f"Normalised NRC-VAD keys: {len(lexicon_lookup):,}")
print(f"Unigram keys: {len(unigram_lookup):,}")
print(f"MWE keys: {len(mwe_lookup):,}")
lexicon_lookup.head()

Normalised NRC-VAD keys: 50,815
Unigram keys: 41,110
MWE keys: 9,705


,lexicon_key,lexicon_length,valence,arousal,dominance,source_terms,source_term_count,lexicon_tuple
0,aaaaaaah,1,-0.042,0.212,-0.418,aaaaaaah,1,"(aaaaaaah,)"
1,aaaah,1,0.040,0.272,-0.436,aaaah,1,"(aaaah,)"
2,aardvark,1,-0.146,-0.020,-0.126,aardvark,1,"(aardvark,)"
3,aback,1,-0.230,-0.186,-0.424,aback,1,"(aback,)"
4,abacus,1,0.020,-0.448,-0.030,abacus,1,"(abacus,)"


## Extract VAD-Matched Collocates

The main index follows Baes' collocate-count logic: focal mention tokens are removed, but stopwords are not removed. Target-form and baseline-form exclusions are named separately so the policy is explicit: only the analysis unit being scored is removed from its own window, and cross-unit affective words remain eligible collocates. Matching is greedy: NRC multi-word expressions are matched first, then remaining unigram lemmas. The resulting VAD table is intentionally reusable for the later Intensity notebook.


In [13]:
TARGET_RAW_FORM_EXCLUSION_TOKENS = {
    "adhd": {"adhd"},
    "attention_deficit": {"attention", "deficit", "hyperactivity", "disorder"},
    "autism": {"autism"},
    "autistic": {"autistic"},
    "autism_spectrum": {"autism", "spectrum"},
    "asd_disambiguated": {"asd"},
}
BASELINE_RAW_FORM_EXCLUSION_TOKENS = {
    "frustration": {"frustration"},
    "sadness": {"sadness"},
    "loneliness": {"loneliness"},
}
RAW_FORM_EXCLUSION_TOKENS_BY_ROLE = {
    "target": TARGET_RAW_FORM_EXCLUSION_TOKENS,
    "baseline": BASELINE_RAW_FORM_EXCLUSION_TOKENS,
}


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def exclusion_tokens_for_row(row: pd.Series) -> set[str]:
    raw_forms = {str(row["raw_form"])} | set(split_pipe_values(row.get("collapsed_raw_forms")))
    role_exclusions = RAW_FORM_EXCLUSION_TOKENS_BY_ROLE.get(str(row["term_role"]), {})
    tokens: set[str] = set()
    for raw_form in raw_forms:
        tokens.update(role_exclusions.get(raw_form, {raw_form.replace("_", " ")}))
    matched_text = str(row.get("matched_text") or "")
    if matched_text:
        tokens.update(part.lower() for part in re.findall(r"[A-Za-z]+", matched_text))
    return tokens


def lexical_tokens(doc, excluded_terms: set[str]) -> list[dict[str, object]]:
    tokens = []
    for token in doc:
        lemma = token.lemma_.lower()
        lower = token.text.lower()
        if not token.is_alpha or token.like_num or len(lemma) <= 1:
            continue
        if lemma in excluded_terms or lower in excluded_terms:
            continue
        tokens.append({"token_index": token.i, "text": token.text, "lemma": lemma})
    return tokens


def match_vad_units(tokens: list[dict[str, object]]) -> list[dict[str, object]]:
    matches: list[dict[str, object]] = []
    index = 0
    while index < len(tokens):
        matched = None
        for length in mwe_lengths:
            if index + length > len(tokens):
                continue
            key = tuple(token["lemma"] for token in tokens[index : index + length])
            if key in mwe_lookup:
                matched = (length, mwe_lookup[key], "mwe")
                break
        if matched is None:
            key = (tokens[index]["lemma"],)
            if key in unigram_lookup:
                matched = (1, unigram_lookup[key], "unigram")
        if matched is None:
            index += 1
            continue
        length, lexicon_row, collocate_type = matched
        span_tokens = tokens[index : index + length]
        matches.append(
            {
                "collocate": lexicon_row.lexicon_key,
                "collocate_type": collocate_type,
                "surface_text": " ".join(str(token["text"]) for token in span_tokens),
                "token_start_in_window": int(span_tokens[0]["token_index"]),
                "token_end_in_window": int(span_tokens[-1]["token_index"] + 1),
                "matched_token_count": int(length),
                "valence": float(lexicon_row.valence),
                "arousal": float(lexicon_row.arousal),
                "dominance": float(lexicon_row.dominance),
                "source_terms": lexicon_row.source_terms,
                "source_term_count": int(lexicon_row.source_term_count),
            }
        )
        index += length
    return matches

match_rows: list[dict[str, object]] = []
coverage_rows: list[dict[str, object]] = []

for doc, (_, row) in zip(nlp.pipe(contexts["token_window_5"].astype(str), batch_size=1000), contexts.iterrows()):
    excluded_terms = exclusion_tokens_for_row(row)
    tokens = lexical_tokens(doc, excluded_terms)
    vad_matches = match_vad_units(tokens)
    matched_token_positions = sum(match["matched_token_count"] for match in vad_matches)

    coverage_rows.append(
        {
            "context_row_id": int(row["context_row_id"]),
            "doc_id": row["doc_id"],
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "raw_form": row["raw_form"],
            "candidate_collocate_tokens": len(tokens),
            "matched_vad_units": len(vad_matches),
            "matched_token_positions": int(matched_token_positions),
            "has_vad_match": bool(vad_matches),
        }
    )

    for match in vad_matches:
        match_rows.append(
            {
                "context_row_id": int(row["context_row_id"]),
                "doc_id": row["doc_id"],
                "lsc_year": int(row["lsc_year"]),
                "published_year": int(row["published_year"]),
                "source_year": int(row["source_year"]),
                "analysis_unit": row["analysis_unit"],
                "term_role": row["term_role"],
                "target_group": row["target_group"],
                "raw_form": row["raw_form"],
                "registered_domain": row["registered_domain"],
                **match,
            }
        )

vad_matches = pd.DataFrame(match_rows)
context_coverage = pd.DataFrame(coverage_rows)
if vad_matches.empty:
    raise RuntimeError("No NRC-VAD collocates matched the shared LSC contexts.")

vad_matches.to_parquet(VAD_MATCHES_PATH, index=False)
context_coverage.to_parquet(VAD_CONTEXT_COVERAGE_PATH, index=False)

print(f"VAD match rows: {len(vad_matches):,}")
print(f"Contexts with any VAD match: {context_coverage['has_vad_match'].sum():,} / {len(context_coverage):,}")
vad_matches.head()


VAD match rows: 925,660
Contexts with any VAD match: 146,309 / 146,471


,context_row_id,doc_id,lsc_year,published_year,source_year,analysis_unit,term_role,target_group,raw_form,registered_domain,collocate,collocate_type,surface_text,token_start_in_window,token_end_in_window,matched_token_count,valence,arousal,dominance,source_terms,source_term_count
0,0,00244c78408b04c9,2014,2014,2014,ADHD,target,ADHD,adhd,ifsw.org,increase,unigram,increase,0,1,1,0.494,0.317,0.659,increase | increased,2
1,0,00244c78408b04c9,2014,2014,2014,ADHD,target,ADHD,adhd,ifsw.org,of,unigram,of,1,2,1,-0.111,-0.119,-0.161,of | of a,2
2,0,00244c78408b04c9,2014,2014,2014,ADHD,target,ADHD,adhd,ifsw.org,medication,unigram,medications,2,3,1,0.230,-0.122,0.000,medication,1
3,0,00244c78408b04c9,2014,2014,2014,ADHD,target,ADHD,adhd,ifsw.org,to,unigram,to,3,4,1,0.110,-0.095,-0.167,to | to me,2
4,0,00244c78408b04c9,2014,2014,2014,ADHD,target,ADHD,adhd,ifsw.org,treat,unigram,treat,4,5,1,0.341,-0.016,-0.002,a treat | treat,2


## Annual Valence Index

Annual valence is the weighted mean of all matched VAD collocate occurrences for each analysis unit and publication year. Coverage is reported alongside the index so sparse or unstable unit-years are not over-interpreted.

In [14]:
coverage = (
    context_coverage.groupby(["lsc_year", "analysis_unit"], as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_vad_units_coverage=("matched_vad_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_vad_match=("has_vad_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_vad_match"] / coverage["context_rows"].replace(0, np.nan)

annual_valence = (
    vad_matches.groupby(["lsc_year", "analysis_unit", "term_role", "target_group"], as_index=False)
    .agg(
        valence_mean=("valence", "mean"),
        valence_sd=("valence", "std"),
        arousal_mean_for_reuse=("arousal", "mean"),
        dominance_mean_for_reuse=("dominance", "mean"),
        matched_vad_units=("valence", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_valence = annual_valence.merge(coverage, on=["lsc_year", "analysis_unit"], how="left")
annual_valence.head()

,lsc_year,analysis_unit,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage
0,2014,ADHD,target,ADHD,0.057990,0.400409,-0.014848,0.024289,9051,1807,935,1587,937,11380,9051,9754,1578,0.857118,0.994329
1,2014,Autism,target,Autism,0.125275,0.371950,-0.034798,0.050380,25840,3511,2373,4185,2373,33503,25840,27632,4184,0.824762,0.999761
2,2014,frustration,baseline,baseline,0.057724,0.350695,-0.023373,0.032570,36480,5210,5108,5688,5108,47353,36480,39601,5688,0.836293,1.000000
3,2014,loneliness,baseline,baseline,0.026362,0.410541,-0.023376,-0.005757,10270,2441,1333,1624,1333,13043,10270,11050,1624,0.847198,1.000000
4,2014,sadness,baseline,baseline,0.045275,0.407450,-0.016308,0.009525,19109,3215,2585,3048,2585,24191,19109,20419,3048,0.844074,1.000000


## Bootstrap Confidence Intervals

Uncertainty is estimated by resampling documents within each analysis-unit year. This respects the fact that collocates are clustered within documents rather than independent token observations.

In [15]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows: list[dict[str, object]] = []

doc_scores = (
    vad_matches.groupby(["lsc_year", "analysis_unit", "doc_id"], as_index=False)
    .agg(valence_sum=("valence", "sum"), matched_vad_units=("valence", "size"))
)

for (year, unit), frame in doc_scores.groupby(["lsc_year", "analysis_unit"], sort=True):
    valence_sums = frame["valence_sum"].to_numpy(dtype=float)
    unit_counts = frame["matched_vad_units"].to_numpy(dtype=float)
    n_docs = len(frame)
    estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        estimates[repetition] = valence_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_rows.append(
        {
            "lsc_year": int(year),
            "analysis_unit": unit,
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "valence_bootstrap_mean": float(np.nanmean(estimates)),
            "valence_ci_low": float(np.nanpercentile(estimates, 2.5)),
            "valence_ci_high": float(np.nanpercentile(estimates, 97.5)),
        }
    )

bootstrap = pd.DataFrame(bootstrap_rows)
annual_valence = annual_valence.merge(bootstrap, on=["lsc_year", "analysis_unit"], how="left")
annual_valence.to_csv(ANNUAL_VALENCE_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)
annual_valence.head()

,lsc_year,analysis_unit,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,bootstrap_repetitions,bootstrap_unit,valence_bootstrap_mean,valence_ci_low,valence_ci_high
0,2014,ADHD,target,ADHD,0.057990,0.400409,-0.014848,0.024289,9051,1807,935,1587,937,11380,9051,9754,1578,0.857118,0.994329,500,doc_id,0.057978,0.047519,0.068167
1,2014,Autism,target,Autism,0.125275,0.371950,-0.034798,0.050380,25840,3511,2373,4185,2373,33503,25840,27632,4184,0.824762,0.999761,500,doc_id,0.125277,0.119619,0.130637
2,2014,frustration,baseline,baseline,0.057724,0.350695,-0.023373,0.032570,36480,5210,5108,5688,5108,47353,36480,39601,5688,0.836293,1.000000,500,doc_id,0.057550,0.053310,0.061547
3,2014,loneliness,baseline,baseline,0.026362,0.410541,-0.023376,-0.005757,10270,2441,1333,1624,1333,13043,10270,11050,1624,0.847198,1.000000,500,doc_id,0.026338,0.016934,0.034917
4,2014,sadness,baseline,baseline,0.045275,0.407450,-0.016308,0.009525,19109,3215,2585,3048,2585,24191,19109,20419,3048,0.844074,1.000000,500,doc_id,0.045247,0.039036,0.051721


## Contributor Diagnostics

The top-collocate table identifies which NRC-VAD matches contribute most to positive or negative annual scores. These diagnostics are not automatic exclusions; they guide manual interpretation and help detect cases where a few words dominate a trajectory.

In [16]:
collocate_counts = (
    vad_matches.groupby(["analysis_unit", "lsc_year", "collocate", "collocate_type"], as_index=False)
    .agg(
        count=("collocate", "size"),
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        dominance=("dominance", "mean"),
        documents=("doc_id", "nunique"),
    )
)
collocate_counts["weighted_valence_contribution"] = collocate_counts["count"] * collocate_counts["valence"]
collocate_counts["abs_weighted_valence_contribution"] = collocate_counts["weighted_valence_contribution"].abs()
collocate_counts["total_matches_for_unit_year"] = collocate_counts.groupby(["analysis_unit", "lsc_year"])["count"].transform("sum")
collocate_counts["match_share"] = collocate_counts["count"] / collocate_counts["total_matches_for_unit_year"]

positive_top = (
    collocate_counts.sort_values(["analysis_unit", "lsc_year", "weighted_valence_contribution"], ascending=[True, True, False])
    .groupby(["analysis_unit", "lsc_year"])
    .head(10)
    .assign(contribution_direction="positive")
)
negative_top = (
    collocate_counts.sort_values(["analysis_unit", "lsc_year", "weighted_valence_contribution"], ascending=[True, True, True])
    .groupby(["analysis_unit", "lsc_year"])
    .head(10)
    .assign(contribution_direction="negative")
)
top_collocates = pd.concat([positive_top, negative_top], ignore_index=True).sort_values(
    ["analysis_unit", "lsc_year", "contribution_direction", "abs_weighted_valence_contribution"],
    ascending=[True, True, True, False],
)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)
top_collocates.head(20)

,analysis_unit,lsc_year,collocate,collocate_type,count,valence,arousal,dominance,documents,weighted_valence_contribution,abs_weighted_valence_contribution,total_matches_for_unit_year,match_share,contribution_direction
650,ADHD,2014,disorder,unigram,128,-0.675000,0.530000,-0.494000,116,-86.400000,86.400000,9051,0.014142,negative
651,ADHD,2014,mental disorder,mwe,67,-0.892000,0.204000,-0.428000,67,-59.764000,59.764000,9051,0.007402,negative
652,ADHD,2014,depression,unigram,62,-0.938000,0.040000,-0.536000,57,-58.156000,58.156000,9051,0.006850,negative
653,ADHD,2014,autism,unigram,95,-0.530000,0.000000,-0.106000,84,-50.350000,50.350000,9051,0.010496,negative
654,ADHD,2014,outrageous,unigram,67,-0.694000,0.890000,0.426000,67,-46.498000,46.498000,9051,0.007402,negative
655,ADHD,2014,diagnose,unigram,81,-0.455500,-0.035500,0.019500,68,-36.895500,36.895500,9051,0.008949,negative
656,ADHD,2014,anxiety,unigram,46,-0.708000,0.730000,-0.316000,45,-32.568000,32.568000,9051,0.005082,negative
657,ADHD,2014,drug,unigram,45,-0.665667,0.542000,-0.425000,42,-29.955000,29.955000,9051,0.004972,negative
658,ADHD,2014,of,unigram,235,-0.111000,-0.119000,-0.161000,192,-26.085000,26.085000,9051,0.025964,negative
659,ADHD,2014,problem,unigram,27,-0.876000,0.288000,-0.170000,25,-23.652000,23.652000,9051,0.002983,negative


## Plots and Audit Flags

The plots use the shared LSC figure style: target groups and comparator terms are faceted rather than overlaid, colours and markers are reserved consistently for each analysis unit, and uncertainty ribbons are shown where bootstrap intervals are available. Audit flags surface coverage or concentration issues that should be checked before using a year-specific sentiment movement as evidence of connotational change.


In [17]:
def save_lsc_figure(fig, png_path: Path) -> Path:
    fig.tight_layout(pad=1.2)
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    return pdf_path


def plot_lsc_panel(ax, frame: pd.DataFrame, units: list[str], y_column: str, *, ribbon: bool = False) -> None:
    for unit in units:
        unit_frame = frame.loc[frame["analysis_unit"] == unit].sort_values("lsc_year")
        ax.plot(
            unit_frame["lsc_year"],
            unit_frame[y_column],
            marker=LSC_UNIT_MARKERS[unit],
            markersize=5,
            linewidth=2.3,
            label=LSC_UNIT_LABELS[unit],
            color=LSC_UNIT_COLORS[unit],
        )
        if ribbon:
            ax.fill_between(
                unit_frame["lsc_year"],
                unit_frame["valence_ci_low"],
                unit_frame["valence_ci_high"],
                color=LSC_UNIT_COLORS[unit],
                alpha=0.15,
                linewidth=0,
            )
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.legend(loc="upper left", ncol=len(units))


fig, axes = plt.subplots(2, 1, figsize=(10.5, 7.2), sharex=True)
fig.suptitle("NRC-VAD Valence of Local LSC Collocates", fontsize=15, fontweight="bold", x=0.02, ha="left")
plot_lsc_panel(axes[0], annual_valence, TARGET_UNITS, "valence_mean", ribbon=True)
plot_lsc_panel(axes[1], annual_valence, BASELINE_UNITS, "valence_mean", ribbon=True)
for ax, title in zip(axes, ["Target groups", "Comparator terms"]):
    ax.axhline(0, color="#263238", linewidth=0.8, alpha=0.65)
    ax.set_title(title, loc="left", fontsize=11.5, fontweight="bold")
    ax.set_ylabel("Mean valence (-1 to 1)")
axes[1].set_xlabel("Publication year")
valence_pdf_path = save_lsc_figure(fig, VALENCE_PLOT_PATH)
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(10.5, 7.2), sharex=True)
fig.suptitle("NRC-VAD Matched-Token Coverage", fontsize=15, fontweight="bold", x=0.02, ha="left")
plot_lsc_panel(axes[0], coverage, TARGET_UNITS, "matched_token_coverage")
plot_lsc_panel(axes[1], coverage, BASELINE_UNITS, "matched_token_coverage")
for ax, title in zip(axes, ["Target groups", "Comparator terms"]):
    ax.set_title(title, loc="left", fontsize=11.5, fontweight="bold")
    ax.set_ylabel("Matched token-position share")
    ax.set_ylim(0.75, 1.01)
axes[1].set_xlabel("Publication year")
coverage_pdf_path = save_lsc_figure(fig, COVERAGE_PLOT_PATH)
plt.close(fig)

flags: list[dict[str, object]] = []
for _, row in coverage.iterrows():
    if row["matched_token_coverage"] < LOW_MATCHED_TOKEN_COVERAGE_WARN:
        flags.append(
            {
                "severity": "warn",
                "category": "low_matched_token_coverage",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "value": float(row["matched_token_coverage"]),
            }
        )
    if row["context_match_coverage"] < LOW_CONTEXT_COVERAGE_WARN:
        flags.append(
            {
                "severity": "warn",
                "category": "low_context_match_coverage",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "value": float(row["context_match_coverage"]),
            }
        )

for _, row in collocate_counts.loc[collocate_counts["match_share"] >= TOP_COLLOCATE_SHARE_WARN].iterrows():
    flags.append(
        {
            "severity": "warn",
            "category": "top_collocate_concentration",
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "value": float(row["match_share"]),
            "detail": row["collocate"],
        }
    )

audit_flags = pd.DataFrame(flags, columns=["severity", "category", "lsc_year", "analysis_unit", "value", "detail"])
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

print(f"Wrote {ANNUAL_VALENCE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {TOP_COLLOCATES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VAD_MATCHES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VAD_CONTEXT_COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VALENCE_PLOT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {valence_pdf_path.relative_to(PROJECT_ROOT)}")
print(f"Wrote {COVERAGE_PLOT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {coverage_pdf_path.relative_to(PROJECT_ROOT)}")
print(f"Audit warnings: {len(audit_flags):,}")
annual_valence.sort_values(["analysis_unit", "lsc_year"]).head(20)


Wrote data/processed/lsc/sentiment/lsc_sentiment_annual_valence.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_coverage.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_top_collocates.csv
Wrote data/processed/lsc/sentiment/lsc_sentiment_audit_flags.csv
Wrote data/interim/lsc/vad/lsc_vad_collocate_matches.parquet
Wrote data/interim/lsc/vad/lsc_vad_context_coverage.parquet
Wrote reports/figures/lsc/sentiment/lsc_sentiment_valence_trajectories.png
Wrote reports/figures/lsc/sentiment/lsc_sentiment_valence_trajectories.pdf
Wrote reports/figures/lsc/sentiment/lsc_sentiment_coverage.png
Wrote reports/figures/lsc/sentiment/lsc_sentiment_coverage.pdf
Audit warnings: 0


,lsc_year,analysis_unit,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,bootstrap_repetitions,bootstrap_unit,valence_bootstrap_mean,valence_ci_low,valence_ci_high
0,2014,ADHD,target,ADHD,0.057990,0.400409,-0.014848,0.024289,9051,1807,935,1587,937,11380,9051,9754,1578,0.857118,0.994329,500,doc_id,0.057978,0.047519,0.068167
5,2015,ADHD,target,ADHD,0.069267,0.391764,-0.030837,0.021069,6636,1537,750,1176,751,8370,6636,7093,1168,0.847431,0.993197,500,doc_id,0.068268,0.055448,0.079592
10,2016,ADHD,target,ADHD,0.066479,0.400620,-0.017718,0.026805,7529,1682,887,1336,888,9518,7529,8034,1330,0.844085,0.995509,500,doc_id,0.066088,0.052889,0.077888
15,2017,ADHD,target,ADHD,0.067613,0.392557,-0.016600,0.021548,7314,1639,847,1297,852,9236,7314,7841,1283,0.848961,0.989206,500,doc_id,0.068045,0.056604,0.078942
20,2018,ADHD,target,ADHD,0.060057,0.396798,-0.025963,0.010486,7697,1711,881,1331,882,9694,7697,8207,1326,0.846606,0.996243,500,doc_id,0.060244,0.048670,0.072306
25,2019,ADHD,target,ADHD,0.056002,0.395419,-0.020784,0.012696,6658,1503,756,1138,759,8285,6658,7080,1132,0.854556,0.994728,500,doc_id,0.056299,0.044083,0.066669
30,2020,ADHD,target,ADHD,0.060758,0.398342,-0.018769,0.017814,6373,1542,742,1132,743,8082,6373,6871,1119,0.850161,0.988516,500,doc_id,0.060873,0.045616,0.074054
35,2021,ADHD,target,ADHD,0.052269,0.398496,-0.020522,0.007437,6591,1624,777,1146,779,8336,6591,7055,1138,0.846329,0.993019,500,doc_id,0.051794,0.040317,0.064351
40,2022,ADHD,target,ADHD,0.077157,0.385535,-0.027879,0.028455,6954,1655,736,1186,739,8642,6954,7450,1176,0.862069,0.991568,500,doc_id,0.076686,0.064090,0.088924
45,2023,ADHD,target,ADHD,0.074312,0.392148,-0.014448,0.034026,5469,1394,588,950,589,6822,5469,5844,935,0.856640,0.984211,500,doc_id,0.073951,0.060283,0.087633


## Compact Handoff Summary

This final table keeps the notebook rerun easy to inspect: it shows the annual valence range, coverage range, and warning count by analysis unit. Substantive interpretation should wait until the manual audit and later cross-dimension synthesis.

In [18]:
handoff_summary = (
    annual_valence.groupby("analysis_unit", as_index=False)
    .agg(
        years=("lsc_year", "nunique"),
        valence_min=("valence_mean", "min"),
        valence_max=("valence_mean", "max"),
        matched_units=("matched_vad_units", "sum"),
        documents_with_matches=("documents_with_matches", "sum"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        min_context_match_coverage=("context_match_coverage", "min"),
    )
)
warning_counts = audit_flags.groupby("analysis_unit").size().rename("warnings").reset_index() if not audit_flags.empty else pd.DataFrame({"analysis_unit": [], "warnings": []})
handoff_summary = handoff_summary.merge(warning_counts, on="analysis_unit", how="left").fillna({"warnings": 0})
handoff_summary["warnings"] = handoff_summary["warnings"].astype(int)
handoff_summary

,analysis_unit,years,valence_min,valence_max,matched_units,documents_with_matches,min_matched_token_coverage,min_context_match_coverage,warnings
0,ADHD,13,0.052269,0.087701,81757,9164,0.844085,0.984211,0
1,Autism,13,0.116996,0.130755,211778,20516,0.824762,0.996661,0
2,frustration,13,0.050803,0.092382,342830,46712,0.836293,0.999522,0
3,loneliness,13,0.013214,0.072420,121231,15093,0.847198,0.999540,0
4,sadness,13,0.021604,0.047176,168064,22601,0.844074,0.998270,0
